playing around with LSH package! (15 videos, one video of me signing 'chocolate' as the query, first 10 frames only of each video!)


In [1]:
from lshashpy3 import LSHash
import pandas as pd
import numpy as np

/opt/miniconda3/envs/asl/lib/python3.10/site-packages/lshashpy3/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Setting up LSH instances
# video --> video

# Load in frame_training_data.csv (product of preprocess_for_LSH.ipynb)

data = pd.read_csv("frame_training_data.csv")
flat_poses_first10 = []
flat_poses_pseudo = []
labels = []
# labels_first10 = []
# labels_pseudo = []
for i, row in data.iterrows():
    points_pseudo = row['flat_pseudo_n7'].strip("[]").split(" ") # load in the limited pseudo-keyframe data: same # frames required for vid-->vid processing
    points_first10 = row['flat_10frames'].strip("[]").split(" ")
    add_to_flat_poses_pseudo = []
    add_to_flat_poses_first10 = []
    for j in points_pseudo:
        if j != "[" and j!= "]" and j != "":
            add_to_flat_poses_pseudo.append(float(j))
    for k in points_first10:
        if k != "[" and k!= "]" and k != "":
            add_to_flat_poses_first10.append(float(k))
    flat_poses_first10.append(np.array(add_to_flat_poses_first10))
    flat_poses_pseudo.append(np.array(add_to_flat_poses_pseudo))
    labels.append(row['LemmaID'])

d_first10 = 990
d_pseudo = 693 # length of each flattened array (aka list) within flat_poses list; CHANGE THIS AS NEEDED

lsh_first10 = LSHash(hash_size=3, input_dim=d_first10, num_hashtables=1, overwrite=False)
lsh_pseudo = LSHash(hash_size=12, input_dim=d_pseudo, num_hashtables=5,overwrite=False)

# batch index
for i in range(len(labels)):
    lsh_first10.index(flat_poses_first10[i], labels[i])
    lsh_pseudo.index(flat_poses_pseudo[i], labels[i])

# lsh_first10
# lsh_pseudo


In [3]:
# Function for analyzing first 10 frames

def test_first10(coord_file): # input is coordinate file from videos
    correct_label = coord_file[:-17] # exclude "_pose-results.txt"
    print("correct label: ", correct_label)
    test_coords = pd.read_csv(coord_file, names=['pose_index', 'x', 'y', 'z'])
    test_10frames = []

    j = 0 # use to get first 10
    for i, row in test_coords.iterrows():
        test_10frames.append(float(row['x']))
        test_10frames.append(float(row['y']))
        test_10frames.append(float(row['z']))
        j += 1
        if j == 330:
            break
    
    # turn to array
    print("test_10frames ", test_10frames)
    test_10frames_array = np.asarray(test_10frames)
    query_vector = test_10frames_array
    print("query vector: ", query_vector)
    print("len query vector: ", len(query_vector))

    top_n = 1
    nn5_first10 = lsh_first10.query(query_vector, num_results=top_n, distance_func="euclidean")
    print("nn5_first10: ", nn5_first10)

    # get labels
    nn5_first10_labels = []
    for i in range(len(nn5_first10)):
        nn5_first10_labels.append(nn5_first10[i][0][1])

    print("top 3 with first 10 frames: ", nn5_first10_labels)

    # evaluate
    if nn5_first10_labels[0] == correct_label:
        result = "direct match"
    elif correct_label in nn5_first10_labels:
        result = "indirect match"
    else:
        result = "incorrect"
    print("result: ", result)
    return correct_label, nn5_first10_labels, result
    

In [4]:
test_correct_label, test_return_labels, test_result = test_first10("chocolate-me_pose-results.txt")


correct label:  chocolate-me
test_10frames  [799.0, 401.0, -1.0590338706970217, 850.0, 325.0, -0.97856467962265, 878.0, 326.0, -0.9789222478866576, 905.0, 327.0, -0.978831946849823, 750.0, 331.0, -0.9889800548553468, 715.0, 332.0, -0.9885373115539552, 682.0, 336.0, -0.9890896081924438, 950.0, 371.0, -0.4750211834907532, 651.0, 377.0, -0.5145180225372314, 857.0, 484.0, -0.8729515075683594, 732.0, 489.0, -0.8843268752098083, 1136.0, 798.0, -0.1097216382622718, 484.0, 789.0, -0.2813276350498199, 1383.0, 1314.0, -0.4277065992355346, 254.0, 1123.0, -0.5554536581039429, 1322.0, 1386.0, -1.2781623601913452, 274.0, 1499.0, -1.497438907623291, 1326.0, 1422.0, -1.4624131917953491, 240.0, 1602.0, -1.7084596157073977, 1285.0, 1367.0, -1.4852826595306396, 305.0, 1551.0, -1.7757694721221924, 1256.0, 1364.0, -1.3214095830917358, 332.0, 1518.0, -1.5558714866638184, 1012.0, 1755.0, -0.0264199674129486, 540.0, 1746.0, 0.0322829410433769, 967.0, 2568.0, -0.1521487683057785, 530.0, 2541.0, -0.214446976780

In [5]:
# run on relevant files, add results to dataframe and write to csv for analysis
vid2vid_first10_results_df = pd.DataFrame(columns=['correct_label', "returned_labels", "evaluation"])


In [6]:
# Function for analyzing pseudo-keyframes (limited)

def test_pseudoKeyframes(coord_file): # input is coordinate file from videos
    correct_label = coord_file[:-17] # exclude "_pose-results.txt"
    print("correct label: ", correct_label)
    test_coords = pd.read_csv(coord_file, names=['pose_index', 'x', 'y', 'z'])
    test_pseudokey = [] # every 10, starting with 0 and ending at 60

    k = 0 # use to get every other 10
    for i, row in test_coords.iterrows():
        if k%10 == 0 or i == 0:
            test_pseudokey.append(float(row['x']))
            test_pseudokey.append(float(row['y']))
            test_pseudokey.append(float(row['z']))
        k += 1
        if len(test_pseudokey) == 693:
            break
    
    # turn to array
    test_pseudokey_array = np.asarray(test_pseudokey)
    query_vector = test_pseudokey_array

    top_n = 5
    nn5_pseudokey = lsh_pseudo.query(query_vector, num_results=top_n, distance_func="euclidean")

    # get labels
    nn5_pseudokey_labels = []
    for i in range(len(nn5_pseudokey)):
        nn5_pseudokey_labels.append(nn5_pseudokey[i][0][1])

    print("top 5 with pseudo-keyframes: ", nn5_pseudokey_labels)

    # evaluate
    if len(nn5_pseudokey_labels) > 0:
        if nn5_pseudokey_labels[0] == correct_label:
            result = "direct match"
        elif correct_label in nn5_pseudokey_labels:
            result = "indirect match"
        else:
            result = "incorrect"
        print("result: ", result)
    else:
        result = "nothing returned"
        print(result)
    return correct_label, result
    

In [7]:
test_correct_label, test_result = test_pseudoKeyframes("chocolate-me_pose-results.txt")

correct label:  chocolate-me
top 5 with pseudo-keyframes:  []
nothing returned


In [8]:
# run on relevant files, add results to dataframe and write to csv for analysis
vid2vid_pseudo_results_df = pd.DataFrame(columns=['correct_label', "returned_labels", "evaluation"])